# Day I13 — 보행 변수: 평균, 변동성, 비대칭, 그리고 속도라는 함정

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I13.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I13.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 165개 기록을 참가자 표 하나로

In [ ]:
import glob, os
import numpy as np, pandas as pd
from gait_events import strides

fs = 100
demo = pd.read_csv("data/gaitpdb/demographics.txt", sep="\t", usecols=range(20))  # 앞 20열만(Day 25)
demo["group"] = demo["Group"].map({1: "PD", 2: "CO"})

def cv(x):
    return 100 * x.std(ddof=1) / x.mean()

rows = []
for path in sorted(glob.glob("data/gaitpdb/*_01.txt")):     # 평소 걷기(walk 01)만
    pid = os.path.basename(path)[:6]                          # 예: GaCo01
    d = np.loadtxt(path)
    feet = {}
    for col, side in [(17, "L"), (18, "R")]:
        r, ok = strides(d[:, col], 20, fs)                    # Day I12의 규칙 그대로
        feet[side] = r[ok]
    swL, swR = feet["L"][:, 3].mean(), feet["R"][:, 3].mean()
    rows.append(dict(
        ID=pid,
        n_strides=min(len(feet["L"]), len(feet["R"])),
        stride_cv=np.mean([cv(feet[s][:, 1]) for s in "LR"]),  # 좌우 발 CV의 평균
        swing_cv=np.mean([cv(feet[s][:, 3]) for s in "LR"]),
        swing_asym=100 * abs(swL - swR) / (0.5 * (swL + swR)), # 대칭 지수(%)
    ))
feat = pd.DataFrame(rows).merge(demo[["ID", "Study", "group", "Speed_01"]], on="ID", how="left")
assert feat["ID"].is_unique and feat["group"].notna().all()   # 한 행 = 한 참가자, 군 정보 빠짐 없음
print(feat.groupby("group").agg(n=("ID", "size"), 최소_stride=("n_strides", "min"),
                                stride_중앙값=("n_strides", "median")))
feat.to_csv("I13_participants.csv", index=False)

### 예시 2 — 군 비교: 효과크기, bootstrap CI, 그리고 속도

In [ ]:
import numpy as np, pandas as pd

df = pd.read_csv("I13_participants.csv")              # 한 행 = 한 참가자(걷기 01 기록 하나)
FEATS = ["stride_cv", "swing_cv", "swing_asym", "Speed_01"]
rng = np.random.default_rng(13)

def cohen_d(a, b):                                # PD − CO, 합동 표준편차
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp

def boot_ci(a, b, stat, B=5000):
    vals = [stat(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(B)]   # 군마다 참가자를 복원 추출
    return np.percentile(vals, [2.5, 97.5])

print(f"{'변수':11s} {'PD 중앙값':>9s} {'CO 중앙값':>9s} {'d (PD−CO)':>9s}  95% bootstrap CI")
for f in FEATS:
    sub = df.dropna(subset=[f])
    a = sub.loc[sub.group == "PD", f].to_numpy()
    b = sub.loc[sub.group == "CO", f].to_numpy()
    lo, hi = boot_ci(a, b, cohen_d)
    print(f"{f:11s} {np.median(a):9.2f} {np.median(b):9.2f} {cohen_d(a, b):9.2f}  [{lo:5.2f}, {hi:5.2f}]  (n = {len(a)}, {len(b)})")

# 속도를 함께 넣으면 군 차이는 얼마나 남나: stride_cv = b0 + b1·PD + b2·속도
sub = df.dropna(subset=["Speed_01"]).reset_index(drop=True)
def pd_coef(s):
    X = np.column_stack([np.ones(len(s)), (s.group == "PD").astype(float), s.Speed_01])
    return np.linalg.lstsq(X, s.stride_cv.to_numpy(), rcond=None)[0]
coef = pd_coef(sub)
boots = np.array([pd_coef(sub.iloc[rng.integers(0, len(sub), len(sub))]) for _ in range(5000)])
lo, hi = np.percentile(boots[:, 1], [2.5, 97.5])
raw = sub.loc[sub.group == "PD", "stride_cv"].mean() - sub.loc[sub.group == "CO", "stride_cv"].mean()
print(f"\nstride_cv 평균 차이(PD − CO): 속도 미보정 {raw:.2f}%p → 속도 보정 {coef[1]:.2f}%p [{lo:.2f}, {hi:.2f}]")
print(f"속도 1 m/s 증가당 stride_cv 변화: {coef[2]:.2f}%p")
for g in ["PD", "CO"]:
    s = sub[sub.group == g]
    print(f"{g}: 속도와 stride_cv의 상관 r = {np.corrcoef(s.Speed_01, s.stride_cv)[0, 1]:.2f} (n = {len(s)})")